# Idea 5 — Watching the Ice Melt: Dataset Downloader
### DSC 106 Final Project · Cryosphere Datasets

This notebook downloads every dataset used across the 6 Idea 5 visualizations directly to your **Desktop**.

| Symbol | Meaning |
|--------|---------|
| 🌐 | Public URL — no account needed |
| 🔑 | Requires free NASA Earthdata account |

**Register here if needed:** https://urs.earthdata.nasa.gov/users/new

Run cells top-to-bottom. Each section is self-contained — skip any you don't need.

---

## Setup — find Desktop & install libraries

In [1]:
import os, sys, subprocess, shutil
from pathlib import Path

def get_desktop():
    home = Path.home()
    for candidate in [home / 'Desktop', home / 'デスクトップ', home / 'Bureau']:
        if candidate.exists():
            return candidate
    fallback = home / 'Desktop'
    fallback.mkdir(exist_ok=True)
    return fallback

DESKTOP = get_desktop()
SAVE_DIR = DESKTOP / 'idea5_cryosphere_data'
SAVE_DIR.mkdir(exist_ok=True)
print(f'Saving all files to:\n  {SAVE_DIR}')

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

ensure('requests')
ensure('tqdm')
import requests
from tqdm.notebook import tqdm
print('Libraries ready')

Saving all files to:
  /Users/joeysandoval/Desktop/idea5_cryosphere_data
Libraries ready


## Shared download helper

In [2]:
def download(url, dest_path, session=None, label=None):
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    label = label or dest_path.name
    req = session or requests
    try:
        with req.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            with open(dest_path, 'wb') as f, tqdm(
                total=total, unit='B', unit_scale=True, desc=label, leave=True
            ) as bar:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
                    bar.update(len(chunk))
        print(f'  Saved -> {dest_path}')
        return dest_path
    except Exception as e:
        print(f'  Failed: {e}')
        return None

---
## 🌐 VIZ 01, 02, 05 — NSIDC Sea Ice Index v3
**Used for:** decadal polar small-multiples · daily spaghetti plot · ice-free projection baseline  
Source: https://nsidc.org/data/g02135/versions/3  
No account required.

In [3]:
nsidc_dir = SAVE_DIR / 'VIZ01_02_05__NSIDC_SeaIceIndex_v3'
nsidc_dir.mkdir(exist_ok=True)

nsidc_files = {
    'N_seaice_extent_monthly_v4.0.csv': (
        'https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/'
        'N_seaice_extent_monthly_v4.0.csv'
    ),
    'S_seaice_extent_monthly_v4.0.csv': (
        'https://noaadata.apps.nsidc.org/NOAA/G02135/south/monthly/data/'
        'S_seaice_extent_monthly_v4.0.csv'
    ),
    'N_seaice_extent_daily_v4.0.csv': (
        'https://noaadata.apps.nsidc.org/NOAA/G02135/north/daily/data/'
        'N_seaice_extent_daily_v4.0.csv'
    ),
    'S_seaice_extent_daily_v4.0.csv': (
        'https://noaadata.apps.nsidc.org/NOAA/G02135/south/daily/data/'
        'S_seaice_extent_daily_v4.0.csv'
    ),
}

print('Downloading NSIDC Sea Ice Index files...')
for filename, url in nsidc_files.items():
    download(url, nsidc_dir / filename, label=filename)
print(f'\nDone -> {nsidc_dir}')

  Failed: 404 Client Error: Not Found for url: https://noaadata.apps.nsidc.org/NOAA/G02135/north/monthly/data/N_seaice_extent_monthly_v4.0.csv
  Failed: 404 Client Error: Not Found for url: https://noaadata.apps.nsidc.org/NOAA/G02135/south/monthly/data/S_seaice_extent_monthly_v4.0.csv


N_seaice_extent_daily_v4.0.csv:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v3/N_seaice_extent_daily_v4.0.csv


S_seaice_extent_daily_v4.0.csv:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v3/S_seaice_extent_daily_v4.0.csv

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v3


---
## 🌐 VIZ 03 — NASA GRACE / GRACE-FO Ice Mass Anomaly
**Used for:** Greenland + Antarctica cumulative mass loss bar chart  
Source: https://ourworldindata.org/grapher/ice-sheet-mass-balance (IMBIE / NASA JPL Tellus composite)  
No account required.

In [ ]:
grace_dir = SAVE_DIR / 'VIZ03__GRACE_IceMassAnomaly'
grace_dir.mkdir(exist_ok=True)

# Our World in Data hosts the IMBIE/GRACE-FO composite as a clean public CSV.
# Columns: Entity (Antarctica|Greenland), Code (ATA|GRL), Day (YYYY-MM-DD), Seasonal variation (Gt cumulative).
owid_url = 'https://ourworldindata.org/grapher/ice-sheet-mass-balance.csv?v=1&csvType=full&useColumnShortNames=false'
combined_path = grace_dir / 'ice_sheet_mass_balance.csv'

print('Downloading combined GRACE/IMBIE ice mass anomaly...')
download(owid_url, combined_path, label='ice_sheet_mass_balance.csv')

# Split into per-hemisphere files for easier downstream use
import csv
rows_grn, rows_ant = [], []
with open(combined_path) as f:
    r = csv.reader(f)
    header = next(r)
    for row in r:
        if not row: continue
        if row[0] == 'Greenland':
            rows_grn.append(row)
        elif row[0] == 'Antarctica':
            rows_ant.append(row)

def _write(path, rows):
    with open(path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['date', 'cumulative_mass_Gt'])
        for row in rows:
            w.writerow([row[2], row[3]])

_write(grace_dir / 'greenland_mass_anomaly.csv', rows_grn)
_write(grace_dir / 'antarctica_mass_anomaly.csv', rows_ant)
print(f'  Greenland records: {len(rows_grn)}')
print(f'  Antarctica records: {len(rows_ant)}')

(grace_dir / 'README.txt').write_text(
    'GRACE / GRACE-FO ice sheet mass anomaly (cumulative Gt since 2002-04-16)\n'
    'Source: Our World in Data composite of IMBIE / NASA JPL Tellus mascon series\n'
    '  https://ourworldindata.org/grapher/ice-sheet-mass-balance\n\n'
    'Files:\n'
    '  ice_sheet_mass_balance.csv  - original (Entity, Code, Day, Seasonal variation)\n'
    '  greenland_mass_anomaly.csv  - cleaned (date, cumulative_mass_Gt)\n'
    '  antarctica_mass_anomaly.csv - cleaned (date, cumulative_mass_Gt)\n\n'
    'For full NetCDF mascon product (needs Earthdata login):\n'
    '  https://podaac.jpl.nasa.gov/dataset/TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.1_V3\n'
)
print(f'\nDone -> {grace_dir}')


---
## 🌐 VIZ 04 — NOAA Arctic Surface Air Temperature Anomaly
**Used for:** Arctic temperature anomaly heatmap (month × year)  
Source: https://arctic.noaa.gov/Report-Card/  
No account required.

In [5]:
noaa_dir = SAVE_DIR / 'VIZ04__NOAA_ArcticTemps'
noaa_dir.mkdir(exist_ok=True)

noaa_url = (
    'https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/regional/time-series/'
    '110/tavg/1/12/1979-2024.csv?base_prd=true&begbaseyear=1981&endbaseyear=2010'
)

print('Downloading NOAA Arctic temperature anomaly...')
download(noaa_url, noaa_dir / 'arctic_60_90N_monthly_temp_anomaly.csv',
         label='arctic_temp_anomaly.csv')

(noaa_dir / 'README.txt').write_text(
    'NOAA NCEI Arctic (60-90N) Surface Air Temperature Anomaly\n'
    'Baseline: 1981-2010\n'
    'Full Arctic Report Card: https://arctic.noaa.gov/Report-Card/\n'
    'MODIS LST monthly (MOD11A3) requires NASA Earthdata login:\n'
    '  https://search.earthdata.nasa.gov/search?q=MOD11A3\n'
)
print(f'\nDone -> {noaa_dir}')

  Failed: 404 Client Error: Not Found for url: https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/regional/time-series/110/tavg/1/12/1979-2024.csv?base_prd=true&begbaseyear=1981&endbaseyear=2010

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ04__NOAA_ArcticTemps


---
## 🌐 VIZ 05 — CMIP6 Arctic Sea Ice Projections (IPCC AR6)
**Used for:** ice-free Arctic summer projection fan chart (SSP1-2.6, SSP2-4.5, SSP5-8.5)  
Files are pre-processed multi-model median series from KNMI Climate Explorer — no account required.  
Full model NetCDFs (several GB each) are on ESGF: https://esgf-node.llnl.gov/search/cmip6/

In [6]:
cmip6_dir = SAVE_DIR / 'VIZ05__CMIP6_SeaIce_Projections'
cmip6_dir.mkdir(exist_ok=True)

knmi_base = 'https://climexp.knmi.nl/CMIP6/Thetis/sea_ice_extent'
cmip6_files = {
    'CMIP6_arctic_sie_ssp126.txt': f'{knmi_base}/ssp126/seaice_extent_north_ssp126_CMIP6_1_1-6.txt',
    'CMIP6_arctic_sie_ssp245.txt': f'{knmi_base}/ssp245/seaice_extent_north_ssp245_CMIP6_1_1-6.txt',
    'CMIP6_arctic_sie_ssp585.txt': f'{knmi_base}/ssp585/seaice_extent_north_ssp585_CMIP6_1_1-6.txt',
}

print('Attempting CMIP6 downloads from KNMI Climate Explorer...')
for filename, url in cmip6_files.items():
    result = download(url, cmip6_dir / filename, label=filename)
    if result is None:
        print(f'  Manual download: https://climexp.knmi.nl/selectfield_cmip6.cgi')

(cmip6_dir / 'README.txt').write_text(
    'CMIP6 Arctic Sea Ice Projections\n'
    'Variable: siconc | Scenarios: SSP1-2.6, SSP2-4.5, SSP5-8.5\n\n'
    'Browser download (no login): https://climexp.knmi.nl/selectfield_cmip6.cgi\n'
    'Full NetCDF via ESGF (free account): https://esgf-node.llnl.gov/search/cmip6/\n'
    '  variable_id=siconc, experiment_id=ssp126|ssp245|ssp585\n'
)
print(f'\nDone -> {cmip6_dir}')

Attempting CMIP6 downloads from KNMI Climate Explorer...
  Failed: 404 Client Error: Not Found for url: https://climexp.knmi.nl/CMIP6/Thetis/sea_ice_extent/ssp126/seaice_extent_north_ssp126_CMIP6_1_1-6.txt
  Manual download: https://climexp.knmi.nl/selectfield_cmip6.cgi
  Failed: 404 Client Error: Not Found for url: https://climexp.knmi.nl/CMIP6/Thetis/sea_ice_extent/ssp245/seaice_extent_north_ssp245_CMIP6_1_1-6.txt
  Manual download: https://climexp.knmi.nl/selectfield_cmip6.cgi
  Failed: 404 Client Error: Not Found for url: https://climexp.knmi.nl/CMIP6/Thetis/sea_ice_extent/ssp585/seaice_extent_north_ssp585_CMIP6_1_1-6.txt
  Manual download: https://climexp.knmi.nl/selectfield_cmip6.cgi

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ05__CMIP6_SeaIce_Projections


---
## 🌐 VIZ 01, 02, 04, 06 — NASA MODIS Imagery via GIBS API
**Used for:** sea ice extent tiles · LST overlays · albedo feedback reference imagery  
No account required. Returns 256×256 PNG tiles at 250 m/pixel.

In [ ]:
gibs_dir = SAVE_DIR / 'VIZ01_02_04_06__MODIS_GIBS_Tiles'
gibs_dir.mkdir(exist_ok=True)

GIBS = (
    'https://gibs.earthdata.nasa.gov/wmts/epsg4326/best/'
    '{layer}/default/{date}/{tms}/{z}/{y}/{x}.{ext}'
)

# (layer, date, tile_matrix_set, file_ext, label)
# zoom=3, y=1, x=4 covers the central Arctic
layers = [
    ('MODIS_Terra_CorrectedReflectance_TrueColor', '2012-09-16', '250m', 'jpg', 'VIZ01_02__2012_record_min'),
    ('MODIS_Terra_CorrectedReflectance_TrueColor', '2023-09-19', '250m', 'jpg', 'VIZ01_02__2023_near_record'),
    ('MODIS_Terra_CorrectedReflectance_TrueColor', '2000-09-15', '250m', 'jpg', 'VIZ01_02__2000_reference'),
    ('MODIS_Terra_Land_Surface_Temp_Day',          '2023-07-15', '1km',  'png', 'VIZ04__2023_summer_LST'),
    ('MODIS_Terra_Land_Surface_Temp_Day',          '2012-07-15', '1km',  'png', 'VIZ04__2012_summer_LST'),
    ('MODIS_Terra_Snow_Cover_Daily',               '2023-09-01', '500m', 'png', 'VIZ06__2023_Sep_albedo'),
    ('MODIS_Terra_Snow_Cover_Daily',               '2000-09-01', '500m', 'png', 'VIZ06__2000_Sep_albedo'),
]

print('Downloading MODIS GIBS tiles...')
for layer, date, tms, ext, label in layers:
    url = GIBS.format(layer=layer, date=date, tms=tms, z=3, y=1, x=4, ext=ext)
    download(url, gibs_dir / f'{label}.{ext}', label=label)

(gibs_dir / 'README.txt').write_text(
    'NASA MODIS Tiles via GIBS API\n'
    'Tile coordinates: zoom=3, y=1, x=4 (central Arctic overview)\n\n'
    'Layer-specific formats:\n'
    '  MODIS_Terra_CorrectedReflectance_TrueColor : 250m, .jpg\n'
    '  MODIS_Terra_Land_Surface_Temp_Day          : 1km,  .png\n'
    '  MODIS_Terra_Snow_Cover_Daily               : 500m, .png\n\n'
    'Custom fetch pattern:\n'
    '  https://gibs.earthdata.nasa.gov/wmts/epsg4326/best/{LAYER}/default/{DATE}/{TMS}/{Z}/{Y}/{X}.{EXT}\n\n'
    'Layer browser: https://worldview.earthdata.nasa.gov/\n'
    'API docs:      https://nasa-gibs.github.io/gibs-api-docs/access-basics/\n'
)
print(f'\nDone -> {gibs_dir}')

VIZ01_02__2012_record_min:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ01_02__2012_record_min.jpg


VIZ01_02__2023_near_record:   0%|          | 0.00/82.2k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ01_02__2023_near_record.jpg


VIZ01_02__2000_reference:   0%|          | 0.00/82.2k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ01_02__2000_reference.jpg


VIZ04__2023_summer_LST:   0%|          | 0.00/38.7k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ04__2023_summer_LST.png


VIZ04__2012_summer_LST:   0%|          | 0.00/42.3k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ04__2012_summer_LST.png
  Failed: 400 Client Error: Bad Request for url: https://gibs.earthdata.nasa.gov/wmts/epsg4326/best/MODIS_Terra_Snow_Cover_Daily/default/2023-09-01/500m/3/1/4.png
  Failed: 400 Client Error: Bad Request for url: https://gibs.earthdata.nasa.gov/wmts/epsg4326/best/MODIS_Terra_Snow_Cover_Daily/default/2000-09-01/500m/3/1/4.png

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles


---
## 🔑 VIZ 01, 03, 04, 06 — NASA Earthdata MODIS HDF Products
**Used for:** MOD29 sea ice · MOD10A1 snow cover · MOD11A3 LST monthly · MCD43A3 albedo  
Requires a **free NASA Earthdata account**: https://urs.earthdata.nasa.gov/users/new  
Uses the `earthaccess` Python library — installs automatically.

In [7]:
# Enter your NASA Earthdata credentials
EARTHDATA_USER = 'jtoast65'   # <- your username
EARTHDATA_PASS = 'Nasadata11!!'   # <- your password

if not EARTHDATA_USER or not EARTHDATA_PASS:
    print('Add your NASA Earthdata credentials above, then re-run.')
    print('Register free at: https://urs.earthdata.nasa.gov/users/new')
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'earthaccess', '-q'])
    import earthaccess
    import os
    os.environ['EARTHDATA_USERNAME'] = EARTHDATA_USER
    os.environ['EARTHDATA_PASSWORD'] = EARTHDATA_PASS
    earthaccess.login(strategy='environment')

    modis_dir = SAVE_DIR / 'VIZ01_03_04_06__MODIS_HDF_Products'
    modis_dir.mkdir(exist_ok=True)

    # (short_name, start_date, end_date, subfolder label)
    products = [
        ('MOD29',   '2023-09-16', '2023-09-16', 'VIZ01_02_05__MOD29_SeaIce'),
        ('MOD10A1', '2023-09-16', '2023-09-16', 'VIZ03__MOD10A1_SnowCover'),
        ('MOD11A3', '2023-09-01', '2023-09-30', 'VIZ04__MOD11A3_LST_Monthly'),
        ('MCD43A3', '2023-09-16', '2023-09-16', 'VIZ06__MCD43A3_Albedo'),
    ]

    for short_name, start, end, label in products:
        print(f'Searching {short_name} ({start})...')
        sub = modis_dir / label
        sub.mkdir(exist_ok=True)
        try:
            results = earthaccess.search_data(
                short_name=short_name,
                temporal=(start, end),
                bounding_box=(-180, 60, 180, 90)
            )
            if results:
                n = min(2, len(results))
                earthaccess.download(results[:n], str(sub))
                print(f'  Downloaded {n} granule(s) -> {sub}')
            else:
                print(f'  No granules found for {short_name} on {start}')
        except Exception as e:
            print(f'  Error: {e}')

    print(f'\nDone -> {modis_dir}')


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Searching MOD29 (2023-09-16)...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

...

  Downloaded 2 granule(s) -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_03_04_06__MODIS_HDF_Products/VIZ03__MOD10A1_SnowCover
Searching MOD11A3 (2023-09-01)...
  No granules found for MOD11A3 on 2023-09-01
Searching MCD43A3 (2023-09-16)...


QUEUEING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2 [00:00<?, ?it/s]

  Downloaded 2 granule(s) -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_03_04_06__MODIS_HDF_Products/VIZ06__MCD43A3_Albedo

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_03_04_06__MODIS_HDF_Products


---
## Summary

In [ ]:
print(f'\n{"="*58}')
print('  Idea 5 Cryosphere Data — Download Summary')
print(f'{"="*58}')
print(f'  Location: {SAVE_DIR}\n')

total = 0
for folder in sorted(SAVE_DIR.iterdir()):
    if folder.is_dir():
        files = [f for f in folder.rglob('*') if f.is_file()]
        sz = sum(f.stat().st_size for f in files)
        total += sz
        sz_str = f'{sz/1024:.1f} KB' if sz < 1e6 else f'{sz/1e6:.1f} MB'
        print(f'  {folder.name}/  ({len(files)} files, {sz_str})')
        for f in sorted(files):
            print(f'    {f.name}')
        print()

t_str = f'{total/1024:.1f} KB' if total < 1e6 else f'{total/1e6:.1f} MB'
print(f'  Total downloaded: {t_str}')
print(f'{"="*58}')